# Curse of Dimensionality

Companion notebook for Sections 1 and 1.2 of the lecture notes.

Objectives:

- visualize why fixed-size samples become sparse in high dimensions;
- simulate distance concentration;
- connect high dimensionality with k-NN behavior;
- separate useful dimensions from noisy dimensions.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True


## 1. Hypercube volume intuition

In a unit interval, covering 10% of the range requires length 0.1. In `d` dimensions, covering 10% of the volume with a small hypercube requires side length `0.1 ** (1 / d)`. The side length quickly approaches 1.


In [ ]:
dims = np.arange(1, 51)
volume_fraction = 0.10
side_length = volume_fraction ** (1 / dims)

fig, ax = plt.subplots()
ax.plot(dims, side_length, marker='o', markersize=3)
ax.set_xlabel('Dimension d')
ax.set_ylabel('Side length needed')
ax.set_title('Side length needed to cover 10% of a unit hypercube')
ax.set_ylim(0, 1.05)
plt.show()

pd.DataFrame({'dimension': [1, 2, 3, 10, 20, 50],
              'side_length_for_10_percent_volume': volume_fraction ** (1 / np.array([1, 2, 3, 10, 20, 50]))})


## 2. Distance concentration

For points sampled uniformly in `[0, 1]^d`, compare the nearest and farthest pairwise distances. As `d` grows, distances often become less distinguishable.


In [ ]:
from sklearn.metrics import pairwise_distances

rng = np.random.default_rng(42)
records = []

for d in range(1, 101):
    X = rng.uniform(size=(250, d))
    D = pairwise_distances(X)
    distances = D[np.triu_indices_from(D, k=1)]
    records.append({
        'dimension': d,
        'min_distance': distances.min(),
        'max_distance': distances.max(),
        'mean_distance': distances.mean(),
        'max_over_min': distances.max() / distances.min(),
        'std_over_mean': distances.std() / distances.mean(),
    })

distance_df = pd.DataFrame(records)
distance_df.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(distance_df['dimension'], distance_df['max_over_min'])
axes[0].set_xlabel('Dimension d')
axes[0].set_ylabel('max distance / min distance')
axes[0].set_title('Nearest and farthest distances become less separated')

axes[1].plot(distance_df['dimension'], distance_df['std_over_mean'])
axes[1].set_xlabel('Dimension d')
axes[1].set_ylabel('std(distance) / mean(distance)')
axes[1].set_title('Relative spread of distances shrinks')

plt.tight_layout()
plt.show()


## 3. Consequence for k-NN

k-NN predicts from nearby training examples. If most points are at similar distances, the notion of nearest neighbor becomes weaker. Here we keep 5 informative features and add increasing numbers of noisy features.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []

for noise_features in [0, 5, 10, 20, 50, 100, 200]:
    X, y = make_classification(
        n_samples=1200,
        n_features=5 + noise_features,
        n_informative=5,
        n_redundant=0,
        n_repeated=0,
        n_classes=2,
        class_sep=1.2,
        random_state=42,
        shuffle=False,
    )
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=7)),
    ])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
    rows.append({'total_features': X.shape[1], 'noise_features': noise_features,
                 'accuracy_mean': scores.mean(), 'accuracy_std': scores.std()})

knn_df = pd.DataFrame(rows)
knn_df


In [ ]:
fig, ax = plt.subplots()
ax.errorbar(knn_df['total_features'], knn_df['accuracy_mean'],
            yerr=knn_df['accuracy_std'], marker='o', capsize=3)
ax.set_xscale('log')
ax.set_xlabel('Total number of features')
ax.set_ylabel('5-fold CV accuracy')
ax.set_title('k-NN degrades as irrelevant dimensions are added')
plt.show()


## 4. Feature selection versus feature extraction

A simple feature-selection baseline keeps only the original informative dimensions. PCA, in contrast, creates new linear combinations. Both reduce dimensionality, but they do not mean the same thing.


In [ ]:
from sklearn.decomposition import PCA

X, y = make_classification(
    n_samples=1200,
    n_features=105,
    n_informative=5,
    n_redundant=0,
    n_repeated=0,
    n_classes=2,
    class_sep=1.2,
    random_state=42,
    shuffle=False,
)

pipelines = {
    'all 105 features': Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=7))]),
    'known informative 5 features': Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=7))]),
    'PCA to 5 components': Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=5)), ('knn', KNeighborsClassifier(n_neighbors=7))]),
}

scores = []
for name, pipe in pipelines.items():
    X_used = X[:, :5] if name == 'known informative 5 features' else X
    cv_scores = cross_val_score(pipe, X_used, y, cv=cv, scoring='accuracy')
    scores.append({'method': name, 'accuracy_mean': cv_scores.mean(), 'accuracy_std': cv_scores.std()})

pd.DataFrame(scores).sort_values('accuracy_mean', ascending=False)


## Takeaways

- In high dimensions, a fixed sample size covers the space poorly.
- Pairwise distances can become less informative, which directly affects distance-based methods such as k-NN.
- Removing irrelevant features and extracting lower-dimensional representations are related but distinct strategies.
